In [2]:
# Instalar dependencias (ejecutar solo una vez)
import subprocess
import sys

packages = ["bcrypt", "pycryptodome", "pandas", "openpyxl", "python-dotenv", "click"]
for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
    
print("✅ Todas las dependencias instaladas")

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


✅ Todas las dependencias instaladas


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [ ]:
# (Ya instalado arriba - opcional si quieres instalar/actualizar manualmente)
# pip install pycryptodome

In [3]:
import pandas as pd
import sys
import os
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

load_dotenv(PROJECT_ROOT / ".env.example")

print("Proyecto:", PROJECT_ROOT)
print("src existe:", SRC_PATH.exists())
print("env existe:", (PROJECT_ROOT / ".env").exists())
print("master password cargada:", bool(os.getenv("CRYPTO_MASTER_PASSWORD")))

from user_manager import UserManager, PasswordValidator, DataValidator
from crypto_security import CryptoManager
from audit_logger import audit_logger
from config import LEVEL1_COLS, LEVEL2_COLS, PUBLIC_COLS, DATABASE_PATH, DATA_ENCRYPTED_PATH

user_manager = UserManager()
crypto_manager = CryptoManager()

print("✅ Sistema de Seguridad Criptográfico Cargado")
print(f"📁 BD de usuarios: {DATABASE_PATH}")
print(f"📁 Datos encriptados: {DATA_ENCRYPTED_PATH}")

2026-04-14 05:15:37 - AuditLogger - INFO - ✅ Operación criptográfica: init_crypto_manager


Proyecto: /Users/alvaropadilla7/Downloads/Cripto-Reto-main
src existe: True
env existe: True
master password cargada: True
✅ Configuración cargada correctamente
✅ Sistema de Seguridad Criptográfico Cargado
📁 BD de usuarios: ./data/usuarios_ong.db
📁 Datos encriptados: ./data/base_encriptada.xlsx


# 1. Gestión de Usuarios - Creación de Admin y Usuarios

Este notebook proporciona una interfaz interactiva para:
- Crear usuarios con roles (admin, analista, público)
- Gestionar contraseñas de forma segura
- Encriptar y desencriptar datos
- Ver registros de auditoría

**Para uso en producción, usa el CLI**: `python -m src.cli interactive`

In [7]:
# Función auxiliar: Crear usuario de forma interactiva
def crear_usuario_interactivo():
    """
    Asistente para crear un nuevo usuario.
    """
    print("\n🆕 Crear Nuevo Usuario")
    print("─" * 50)
    
    user_id = input("ID del usuario (ej: usuario_prueba): ").strip()
    
    # Validar ID
    if not DataValidator.validate_user_id(user_id):
        print("❌ ID inválido (3-50 caracteres alfanuméricos)")
        return
    
    # Mostrar requisitos de contraseña
    print("\n📋 Requisitos de Contraseña:")
    print("  • Mínimo 12 caracteres")
    print("  • Al menos 1 MAYÚSCULA, 1 minúscula, 1 número, 1 especial")
    print("  • Ejemplo: Admin@2024!Secure")
    
    password = input("Contraseña: ").strip()
    password_confirm = input("Confirmar contraseña: ").strip()
    
    if password != password_confirm:
        print("❌ Las contraseñas no coinciden")
        return
    
    # Mostrar roles disponibles
    print("\n👤 Roles disponibles:")
    print("  1. admin    - Acceso total")
    print("  2. analista - Ver datos nivel 1")
    print("  3. publico  - Solo datos públicos")
    
    rol_choice = input("Selecciona rol (1-3) [2]: ").strip() or "2"
    roles_map = {"1": "admin", "2": "analista", "3": "publico"}
    rol = roles_map.get(rol_choice, "analista")
    
    # Crear usuario
    try:
        user_manager.create_user(
            user_id,
            password,
            rol,
            created_by="SISTEMA",
            is_primary_admin=(user_id == "admin_master")
        )
        print(f"✅ Usuario '{user_id}' creado exitosamente (rol: {rol})")
    except Exception as e:
        print(f"❌ Error: {e}")

# Ejecutar
crear_usuario_interactivo()


🆕 Crear Nuevo Usuario
──────────────────────────────────────────────────

📋 Requisitos de Contraseña:
  • Mínimo 12 caracteres
  • Al menos 1 MAYÚSCULA, 1 minúscula, 1 número, 1 especial
  • Ejemplo: Admin@2024!Secure

👤 Roles disponibles:
  1. admin    - Acceso total
  2. analista - Ver datos nivel 1
  3. publico  - Solo datos públicos


2026-04-14 05:20:06 - AuditLogger - INFO - 🆕 Nuevo usuario creado por SISTEMA: usuario_prueba (rol: analista)


✅ Usuario 'usuario_prueba' creado exitosamente (rol: analista)


In [ ]:
def ver_certificado_usuario(user_id):
    cert = user_manager.get_active_certificate(user_id)
    if not cert:
        print("No tiene certificado activo")
        return

    print("Certificado activo:")
    print(f"ID: {cert[0]}")
    print(f"Fingerprint: {cert[1]}")
    print(f"Estado: {cert[2]}")
    print(f"Asignado en: {cert[3]}")
    print(f"Expira en: {cert[4]}")

In [ ]:
def asignar_certificado_interactivo():
    print("\n📜 Asignar certificado a usuario")
    print("-" * 40)

    assigned_by = input("Admin principal que asigna: ").strip()
    user_id = input("Usuario destino: ").strip()
    certificate_pem = input("Contenido o identificador del certificado: ").strip()
    expires_at = input("Fecha de expiración (opcional, formato YYYY-MM-DD): ").strip() or None

    try:
        cert_id = user_manager.assign_certificate(
            user_id=user_id,
            certificate_pem=certificate_pem,
            assigned_by=assigned_by,
            expires_at=expires_at
        )
        print(f"✅ Certificado asignado correctamente: {cert_id}")
    except Exception as e:
        print(f"❌ Error: {e}")

In [ ]:
def revocar_certificado_interactivo():
    print("\n🚫 Revocar certificado")
    print("-" * 40)

    revoked_by = input("Admin principal que revoca: ").strip()
    user_id = input("Usuario al que se le revoca: ").strip()

    try:
        user_manager.revoke_certificate(user_id=user_id, revoked_by=revoked_by)
        print("✅ Certificado revocado correctamente")
    except Exception as e:
        print(f"❌ Error: {e}")

In [ ]:
def ver_usuarios_no_cifrados():
    print("\n👑 Ver usuarios no cifrados")
    print("-" * 40)

    requested_by = input("Admin principal: ").strip()

    try:
        usuarios = user_manager.list_raw_users(requested_by)
        if not usuarios:
            print("No hay usuarios registrados")
            return

        for u in usuarios:
            print(u)
    except Exception as e:
        print(f"❌ Error: {e}")

In [ ]:
def rotar_clave_interactivo():
    print("\n🔑 Rotación de clave")
    print("-" * 40)

    key_name = input("Nombre de la clave (ej. basica/admin): ").strip()
    rotated_by = input("Quién rota la clave: ").strip() or "SISTEMA"

    try:
        version = crypto_manager.rotate_key(key_name, rotated_by=rotated_by)
        print(f"✅ Clave rotada correctamente. Nueva versión: v{version}")
    except Exception as e:
        print(f"❌ Error: {e}")

In [ ]:
# Función auxiliar: Login de usuario
def login_usuario():
    """
    Asistente para login interactivo.
    """
    print("\n🔐 Iniciar Sesión")
    print("─" * 50)
    
    user_id = input("ID de usuario: ").strip()
    password = input("Contraseña: ").strip()
    
    try:
        success, rol = user_manager.authenticate(user_id, password)
        print(f"✅ Bienvenido {user_id}!")
        print(f"   Rol: {rol}")
        return user_id, rol
    except Exception as e:
        print(f"❌ Error de autenticación: {e}")
        return None, None

# Ejemplo: crear primer admin
print("\n========================================")
print("Crear Primer Usuario Admin")
print("========================================")
print("Si es la primera vez, crea un admin:")
print("")
print("ID: admin_master")
print("Contraseña: Admin@2024!Inicial (CAMBIAR SEGÚN .env)")
print("Rol: admin")

try:
    # Intentar crear admin (si ya existe, continuará)
    user_manager.create_user(
        "admin_master",
        "Admin@2024!Inicial",
        "admin",
        created_by="SISTEMA"
    )
    print("✅ Admin creado")
except Exception as e:
    print(f"ℹ️  Admin ya existe: {e}")


# 2. Cifrado Base - Encriptación de Datos

Usa la clase `CryptoManager` para encriptar y desencriptar datos de forma segura con derivación PBKDF2 y AES-256-GCM.

In [ ]:
# Las claves se derivan automáticamente de la contraseña maestra (CRYPTO_MASTER_PASSWORD)
# y se almacenan en la BD para reproducibilidad

print("🔑 Gestión Automática de Claves")
print("─" * 50)
print("Las claves se derivan usando PBKDF2 con:")
print("  • Contraseña maestra: CRYPTO_MASTER_PASSWORD (env var)")
print("  • Salt: Almacenado en BD (tabla key_metadata)")
print("  • Iteraciones: 100,000 (OWASP recomendado)")
print("  • Resultado: Reproducible y seguro")
print("")

# Obtener o crear claves
key_basica = crypto_manager.get_or_create_key("basica")
key_admin = crypto_manager.get_or_create_key("admin")

print("✅ Claves derivadas:")
print(f"  • Clave básica (analista): {len(key_basica)} bytes")
print(f"  • Clave admin: {len(key_admin)} bytes")

In [ ]:
def login():
    user_id = input("ID de Usuario: ")
    password = input("Contraseña: ")

    conn = sqlite3.connect('usuarios_ong.db')
    c = conn.cursor()
    c.execute("SELECT password_hash, rol FROM usuarios WHERE id_usuario = ?", (user_id,))
    result = c.fetchone()

    if result and bcrypt.checkpw(password.encode(), result[0]):
        rol = result[1]
        # REGISTRO DE AUDITORÍA
        c.execute("INSERT INTO auditoria VALUES (?, ?, ?)",
                  (user_id, "Inicio de sesión exitoso", datetime.now()))
        conn.commit()
        conn.close()
        print(f"Bienvenido. Rol: {rol}")
        return user_id, rol
    else:
        print("Credenciales incorrectas.")
        conn.close()
        return None, None

#Cifrado Base

In [ ]:
# Columnas según nivel de acceso (ya definidas en config.py)
print("📊 Niveles de Acceso y Columnas Encriptadas")
print("─" * 50)
print(f"\n🟢 Nivel 1 (Analista):\n  {LEVEL1_COLS}")
print(f"\n🔴 Nivel 2 (Admin):\n  {LEVEL2_COLS}")
print(f"\n🔓 Público (Todos):\n  {PUBLIC_COLS}")

In [ ]:
# Las funciones de encriptación/desencriptación están en crypto_manager
# Aquí solo usamos la interfaz simplificada

print("🔐 Encriptación/Desencriptación")
print("─" * 50)
print("Métodos disponibles en CryptoManager:")
print("")
print("1. encrypt_value(valor, clave) → valor encriptado en base64")
print("2. decrypt_value(ciphertext, clave) → valor original")
print("3. encrypt_dataframe(df, level1_cols, level2_cols)")
print("4. decrypt_dataframe(df, level1_cols, level2_cols)")
print("")

# Ejemplo: encriptar un valor
ejemplo_valor = "María García López"
clave = crypto_manager.get_or_create_key("admin")
encriptado = crypto_manager.encrypt_value(ejemplo_valor, clave)
desencriptado = crypto_manager.decrypt_value(encriptado, clave)

print(f"✅ Ejemplo de encriptación:")
print(f"  Original:     {ejemplo_valor}")
print(f"  Encriptado:   {encriptado[:50]}...")
print(f"  Desencriptado: {desencriptado}")

In [ ]:
# Función para encriptar un DataFrame entero
def encriptar_archivo(archivo_excel):
    """
    Encripta un archivo Excel según niveles de acceso.
    """
    try:
        df = pd.read_excel(archivo_excel)
        print(f"📖 Archivo cargado: {archivo_excel}")
        print(f"   Filas: {len(df)}, Columnas: {len(df.columns)}")
        
        df_enc = crypto_manager.encrypt_dataframe(df, LEVEL1_COLS, LEVEL2_COLS)
        
        # Guardar
        output_file = "base_encriptada.xlsx"
        df_enc.to_excel(output_file, index=False)
        
        print(f"✅ Archivo encriptado guardado: {output_file}")
        audit_logger.log_data_access("USUARIO", "admin", "encrypt_data", f"Archivo: {archivo_excel}")
        
        return df_enc
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

# Nota: Descomenta la siguiente línea si tienes un archivo Excel
# df_encrypted = encriptar_archivo("Crypto, prueba libreria.xlsx")

In [ ]:
# Cargar archivo Excel (si existe)
try:
    df = pd.read_excel("Crypto, prueba libreria.xlsx")
    print(f"✅ Archivo cargado: Crypto, prueba libreria.xlsx")
    print(f"   Shape: {df.shape}")
    display(df.head())
except FileNotFoundError:
    print("ℹ️  Archivo 'Crypto, prueba libreria.xlsx' no encontrado")
    print("   Usa encriptar_archivo() cuando tengas el archivo")

In [ ]:
# Guardado automático cuando usas encriptar_archivo()
# df_encrypted.to_excel("base_encriptada.xlsx", index=False)
print("Usa encriptar_archivo() para guardar datos encriptados")

In [ ]:
# Funciones de filtrado según rol
def view_public(df):
    """Vista pública: solo columnas públicas."""
    cols = ["ID Usuario", "Fecha de Ingreso", "País Origen"]
    return df[[c for c in cols if c in df.columns]]

def view_analyst(df):
    """Vista analista: nivel 1 + público."""
    df_view = df.copy()
    for col in LEVEL1_COLS:
        if col in df_view.columns:
            df_view[col] = df_view[col].apply(
                lambda x: crypto_manager.decrypt_value(x, crypto_manager.get_or_create_key("basica")) if x else None
            )
    
    # Mantener solo columnas permitidas
    cols_permitidas = PUBLIC_COLS + LEVEL1_COLS
    return df_view[[c for c in cols_permitidas if c in df_view.columns]]

def view_admin(df):
    """Vista admin: todos los niveles."""
    df_view = df.copy()
    
    for col in LEVEL1_COLS:
        if col in df_view.columns:
            df_view[col] = df_view[col].apply(
                lambda x: crypto_manager.decrypt_value(x, crypto_manager.get_or_create_key("basica")) if x else None
            )
    
    for col in LEVEL2_COLS:
        if col in df_view.columns:
            df_view[col] = df_view[col].apply(
                lambda x: crypto_manager.decrypt_value(x, crypto_manager.get_or_create_key("admin")) if x else None
            )
    
    return df_view

print("✅ Funciones de control de acceso definidas:")
print("  • view_public(df)")
print("  • view_analyst(df)")
print("  • view_admin(df)")

In [ ]:
# Función para obtener vista segura según rol del usuario
def vista_segura(df_encriptado, user_id, rol):
    """
    Devuelve vista del DataFrame según rol del usuario.
    
    Args:
        df_encriptado: DataFrame con datos encriptados
        user_id: ID del usuario solicitando acceso
        rol: Rol del usuario (admin, analista, publico)
    """
    audit_logger.log_data_access(user_id, rol, "view_data", f"Acceso a vista {rol}")
    
    if rol == "admin":
        return view_admin(df_encriptado)
    elif rol == "analista":
        return view_analyst(df_encriptado)
    else:
        return view_public(df_encriptado)

# Función para obtener auditoría
def ver_auditoria(user_id=None, limit=50):
    """Muestra registro de auditoría."""
    records = audit_logger.get_audit_trail(user_id, limit)
    
    if not records:
        print("No hay registros de auditoría")
        return None
    
    df_audit = pd.DataFrame(
        records,
        columns=["ID", "Usuario", "Acción", "Resultado", "Detalles", "Fecha"]
    )
    return df_audit

print("✅ Funciones de acceso seguro definidas:")
print("  • vista_segura(df, user_id, rol)")
print("  • ver_auditoria(user_id=None, limit=50)")

In [ ]:
# Ejemplo de uso: acceso según rol
print("\n╔════════════════════════════════════════════════════════╗")
print("║          DEMOSTRACIÓN DE CONTROL DE ACCESO              ║")
print("╚════════════════════════════════════════════════════════╝")
print("")

# Nota: Para probar, necesitas un archivo encriptado
# Primero ejecuta encriptar_archivo() si tienes un archivo Excel

print("\n📌 Ejemplo de Uso:")
print("─" * 50)
print("")
print("1. Admin ve TODOS los datos (nivel 1 + 2 + público)")
print("")
print("2. Analista ve nivel 1 + público (nombres NO visibles)")
print("")
print("3. Usuario público ve SOLO datos públicos")
print("")
print("Todos los accesos se registran automáticamente en auditoría")
print("")

# Ejemplo si existiera un DataFrame encriptado:
# df_secure = pd.read_excel("base_encriptada.xlsx")
# 
# print("✅ Acceso Admin:")
# display(vista_segura(df_secure, "admin_master", "admin").head())
# 
# print("\n✅ Acceso Analista:")
# display(vista_segura(df_secure, "usuario_analista", "analista").head())

## 3. Almacenamiento y Deployment

### Almacenamiento Local
- **Usuario BD**: `data/usuarios_ong.db` (SQLite)
- **Datos Encriptados**: `data/base_encriptada.xlsx` (Excel)
- **Logs**: `logs/audit.log` (Archivo de texto)

Todos estos archivos están excluidos de Git (`.gitignore`).

### Deploy en Hostgator
Ver `README.md` para instrucciones completas de:
1. Setup en servidor
2. Configuración de variables de entorno
3. Backups automáticos
4. Seguridad en producción

## 4. Carga de Datos

Los usuarios subirán datos mediante:
- **Formularios web** (futuro)
- **Archivo Excel** (actual)

Los datos se encriptan automáticamente antes de guardarse.

In [ ]:
# Ver DataFrame encriptado si existe
try:
    df_secure = pd.read_excel("base_encriptada.xlsx")
    print(f"📊 Vista de datos encriptados:")
    print(f"   Filas: {len(df_secure)}, Columnas: {len(df_secure.columns)}")
    print("")
    display(df_secure.head())
except FileNotFoundError:
    print("ℹ️  No hay archivo base_encriptada.xlsx")
    print("   Ejecuta encriptar_archivo() primero")

## 5. Gestión de Base de Datos Local

La BD se guarda localmente en `data/usuarios_ong.db` (SQLite).

In [ ]:
# Función para mostrar BD de usuarios
def mostrar_base_usuarios():
    """Muestra información de usuarios y auditoría."""
    import sqlite3
    
    try:
        conn = sqlite3.connect(DATABASE_PATH)
        
        print("=" * 80)
        print("🔐 TABLA DE USUARIOS")
        print("=" * 80)
        
        df_usuarios = pd.read_sql_query(
            "SELECT id_usuario, rol, created_at, last_login FROM usuarios ORDER BY created_at DESC",
            conn
        )
        
        if len(df_usuarios) > 0:
            display(df_usuarios)
        else:
            print("(No hay usuarios aún)")
        
        print("\n" + "=" * 80)
        print("📋 REGISTRO DE AUDITORÍA (últimos 20 eventos)")
        print("=" * 80)
        
        df_logs = pd.read_sql_query(
            """
            SELECT id_usuario, accion, resultado, detalles, fecha 
            FROM auditoria 
            ORDER BY fecha DESC 
            LIMIT 20
            """,
            conn
        )
        
        if len(df_logs) > 0:
            display(df_logs)
        else:
            print("(No hay registros de auditoría aún)")
        
        conn.close()
        
    except Exception as e:
        print(f"❌ Error: {e}")

# Ejecutar
print("\n📊 Estado Actual del Sistema")
print("─" * 80)
mostrar_base_usuarios()

In [4]:
print(user_manager.is_primary_admin("admin_master"))

True


In [8]:
cert_id = user_manager.assign_certificate(
    user_id="usuario_prueba",
    certificate_pem="CERT-USUARIO-PRUEBA-001",
    assigned_by="admin_master",
    expires_at="2027-12-31"
)
print(cert_id)

2026-04-14 05:20:18 - AuditLogger - INFO - 📜 CERT assign usuario=usuario_prueba certificado=CERT-usuario_prueba-7c3f0755 - fingerprint=038acdb7d8acd6dda66315096289cb27590e24a190c337c3c47bd0aa92c99cdf


CERT-usuario_prueba-7c3f0755


In [9]:
print(user_manager.get_active_certificate("usuario_prueba"))

('CERT-usuario_prueba-7c3f0755', '038acdb7d8acd6dda66315096289cb27590e24a190c337c3c47bd0aa92c99cdf', 'activo', '2026-04-14 11:20:18', '2027-12-31')


In [10]:
print(user_manager.revoke_certificate("usuario_prueba", "admin_master"))

2026-04-14 05:20:39 - AuditLogger - INFO - 📜 CERT revoke usuario=usuario_prueba certificado=N/A - Certificado activo revocado


True


In [11]:
user_manager.assign_certificate(
    user_id="usuario_prueba",
    certificate_pem="CERT-USUARIO-PRUEBA-001",
    assigned_by="admin_master",
    expires_at="2027-12-31"
)

2026-04-14 07:07:26 - AuditLogger - INFO - 📜 CERT assign usuario=usuario_prueba certificado=CERT-usuario_prueba-3c69b49a - fingerprint=038acdb7d8acd6dda66315096289cb27590e24a190c337c3c47bd0aa92c99cdf


'CERT-usuario_prueba-3c69b49a'